# Q-Energy: Grover's search for a smart appliance schedule

Run each cell in order (Shift + Enter). In Google Colab: **Runtime → Run all**.

**The problem:** 4 appliances must each run once a day in one of 4 time slots. Find a plan whose daily bill is at most ₹45 and that never draws more than 4 kW at once.

In [ ]:
# Step 0: install Qiskit (takes about a minute in Colab)
!pip install -q qiskit qiskit-aer matplotlib pylatexenc

## Step 1: the household
Change any number here and run the cells below again.

In [ ]:
SLOTS = ["Morning", "Noon", "Evening", "Night"]   # codes 00, 01, 10, 11
PRICE = [6, 4, 10, 7]            # ₹ per kWh in each slot

APPLIANCES = [                   # name, power in kW, slots it may use
    ("Washing machine", 0.8, ["Morning", "Noon", "Evening"]),
    ("Dishwasher",      1.2, ["Noon", "Evening", "Night"]),
    ("Water heater",    2.0, ["Morning", "Noon", "Night"]),
    ("EV charger",      3.5, ["Evening", "Night"]),
]

MAX_BILL = 45     # ₹ per day
MAX_PEAK = 4.0    # kW at the same time

## Step 2: turn qubits into a plan
Each appliance uses 2 qubits for its time slot, so 4 appliances need **8 qubits** and there are **2⁸ = 256** possible plans.
`01 01 00 11` means Washing machine → Noon, Dishwasher → Noon, Water heater → Morning, EV → Night.

In [ ]:
N_QUBITS = 2 * len(APPLIANCES)

def decode(bitstring):
    """'01010011' -> ['Noon', 'Noon', 'Morning', 'Night']"""
    return [SLOTS[int(bitstring[i:i + 2], 2)] for i in range(0, N_QUBITS, 2)]

def bill_and_peak(plan):
    load = {s: 0.0 for s in SLOTS}
    for (name, kw, _), slot in zip(APPLIANCES, plan):
        load[slot] += kw
    bill = sum(load[s] * PRICE[i] for i, s in enumerate(SLOTS))
    return bill, max(load.values())

def is_good(plan):
    allowed = all(slot in ok for (_, _, ok), slot in zip(APPLIANCES, plan))
    bill, peak = bill_and_peak(plan)
    return allowed and bill <= MAX_BILL and peak <= MAX_PEAK

print(decode("01010011"), bill_and_peak(decode("01010011")))

## Step 3: find which plans the oracle should mark
The oracle needs to recognise good plans. Here the computer checks each plan's rules (a classical check), and the quantum circuit does the searching. This is a **hybrid quantum-classical** design.

In [ ]:
good = [format(x, f"0{N_QUBITS}b") for x in range(2 ** N_QUBITS) if is_good(decode(format(x, f"0{N_QUBITS}b")))]
N, M = 2 ** N_QUBITS, len(good)
print(f"{M} good plans out of {N}")
for g in good:
    print(" ", g, decode(g), "bill ₹%.1f, peak %.1f kW" % bill_and_peak(decode(g)))

## Step 4: build Grover's circuit
* **H on every qubit:** all 256 plans in equal superposition.
* **Oracle:** flips the sign of each good plan (X gates turn that plan into |11111111⟩, a multi-controlled Z flips it, X undoes).
* **Diffuser:** reflects every amplitude about the average, so the flipped plans grow.
* Repeat **k ≈ π/4 · √(N/M)** times, then measure.

In [ ]:
import math
from qiskit import QuantumCircuit
from qiskit.circuit.library import ZGate

def flip_all_ones(qc):
    """Multi-controlled Z: flips the sign of |11...1> only."""
    qc.append(ZGate().control(N_QUBITS - 1, annotated=False), range(N_QUBITS))

def oracle():
    qc = QuantumCircuit(N_QUBITS, name="Oracle")
    for plan in good:
        zeros = [q for q in range(N_QUBITS) if plan[N_QUBITS - 1 - q] == "0"]  # Qiskit reads right to left
        if zeros: qc.x(zeros)
        flip_all_ones(qc)
        if zeros: qc.x(zeros)
    return qc.to_gate()

def diffuser():
    qc = QuantumCircuit(N_QUBITS, name="Diffuser", global_phase=math.pi)  # exact 2|s><s| - I
    qc.h(range(N_QUBITS)); qc.x(range(N_QUBITS))
    flip_all_ones(qc)
    qc.x(range(N_QUBITS)); qc.h(range(N_QUBITS))
    return qc.to_gate()

k = math.floor(math.pi / (4 * math.asin(math.sqrt(M / N))))
grover = QuantumCircuit(N_QUBITS, N_QUBITS)
grover.h(range(N_QUBITS))
for _ in range(k):
    grover.append(oracle(), range(N_QUBITS))
    grover.append(diffuser(), range(N_QUBITS))
grover.measure(range(N_QUBITS), range(N_QUBITS))
print("Grover rounds k =", k)
grover.draw("mpl", fold=40)

## Step 5: run it on the quantum simulator

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator

sim = AerSimulator(seed_simulator=42)
counts = sim.run(transpile(grover, sim), shots=1000).result().get_counts()

hits = sum(c for b, c in counts.items() if b in good)
print(f"Good plan measured in {hits / 10:.1f}% of 1000 runs (random guessing: {100 * M / N:.1f}%)")
best = max(counts, key=counts.get)
print("Most frequent result:", best, "->", decode(best), "bill ₹%.1f, peak %.1f kW" % bill_and_peak(decode(best)))

In [ ]:
from qiskit.visualization import plot_histogram
top = dict(sorted(counts.items(), key=lambda kv: -kv[1])[:12])
plot_histogram(top, title="Most measured plans (good plans dominate)")

## Step 6: compare with the usual routine

In [ ]:
usual = ["Evening", "Evening", "Morning", "Evening"]
b_u, p_u = bill_and_peak(usual)
b_q, p_q = bill_and_peak(decode(best))
print(f"Usual routine : ₹{b_u:.1f}/day, peak {p_u:.1f} kW")
print(f"Quantum plan  : ₹{b_q:.1f}/day, peak {p_q:.1f} kW")
print(f"Saving        : about ₹{(b_u - b_q) * 30:.0f} a month")

**What to say about it:** checking plans one at a time needs about N/M tries; Grover needs about √(N/M) rounds, a quadratic saving. In this small example the oracle is built from a classical check, and a simulator is not faster than a normal loop. The point is to show the method, which pays off on much larger search spaces and on real quantum hardware.